In [ ]:
import os
import cv2
import json
import numpy as np
from tqdm import tqdm
from IPython.display import Video

In [ ]:
celeb_meta = json.load(open("/home/user/datasets/celebvhq_info.json"))

In [ ]:
meta_info = celeb_meta["meta_info"]
clips = celeb_meta["clips"]

In [ ]:
meta_info.keys()

In [ ]:
appearance_mapping = meta_info["appearance_mapping"]
action_mapping = meta_info["action_mapping"]

In [ ]:
action_mapping

In [ ]:
len(meta_info["appearance_mapping"])

In [ ]:
len(meta_info["action_mapping"])

In [ ]:
len(clips.keys())

In [ ]:
Video("/home/user/datasets/celebs/M2Ohb0FAaJU_1.mp4")

In [ ]:
clips["M2Ohb0FAaJU_1"].keys()

In [ ]:
clips["M2Ohb0FAaJU_1"]["attributes"]

In [ ]:
ans = []
for idx, l in enumerate(clips["M2Ohb0FAaJU_1"]["attributes"]["appearance"]):
    if l:
        ans.append(appearance_mapping[idx])

ans

In [ ]:
ans = []
for idx, l in enumerate(clips["M2Ohb0FAaJU_1"]["attributes"]["action"]):
    if l:
        ans.append(action_mapping[idx])

ans

In [ ]:
appearance = []
actions = []
for key in tqdm(clips.keys()):
    appearance.append(clips[key]["attributes"]["appearance"])
    actions.append(clips[key]["attributes"]["action"])

In [ ]:
np_array = np.array(appearance)
result = np.sum(np_array, axis=0)
result_list = result.tolist()

In [ ]:
result_list

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
  model="gpt-5-nano",
  input=batched["sVZLKLWFDYs_1"],
  text={
    "format": {
      "type": "text"
    },
    "verbosity": "low"
  },
  reasoning={
    "effort": "minimal"
  },
  tools=[],
  store=True,
  include=[]
)

In [ ]:
response

In [ ]:
default_input = [
    {
      "role": "developer",
      "content": [
        {
          "type": "input_text",
          "text": "Developer: # Role and Objective\n- Generate a concise, natural-language Stable Diffusion 2.1-style prompt describing the image, using available appearance and action traits.\n\n# Instructions\n- Begin with a concise checklist (3-7 bullets) of planned steps for analyzing and converting traits into a prompt.\n- Analyze the image and provided traits to summarize the subject, actions, and setting in a single comma-separated prompt that is natural and user-written. Return structured JSON with required and optional keys only.\n\n## Sub-categories\n- **Token Conversion**: Convert snake_case (e.g., \"pale_skin\") to natural language. Map core traits to visually descriptive terms, e.g., \"male\" → \"man\", \"young\" → \"young\", \"bald\" → \"bald\".\n- **Action Conversion**: Change action tokens to gerunds (e.g., \"drink\" → \"drinking\").\n- **Prioritization**: Start with the subject, then action, followed by setting and mood/composition if clearly visible. Prioritize the most visually prominent traits. Keep prompt length within 15–40 words. Keep it really simple.\n- **Defects Handling**: Only include true visual defects (e.g., \"blurry\", \"low resolution\") in \"negative_prompt\". Exclude non-visual or irrelevant traits.\n- **Ambiguity**: If a trait or information is unclear, describe only what is clearly seen. If nothing is certain, return an empty prompt string.\n\n# Context\n- **Input**: Image, appearance_traits[], action_traits[]. Some inputs may be ambiguous or incomplete.\n- **Output**: JSON containing only \"prompt\" (required) and, if visual defects are present, \"negative_prompt\" (optional).\n- **Sample Output**:\n  ```json\n  {\n    \"prompt\": \"young man with pale skin and rosy cheeks, oval face, drinking from a mug at a cafe table, close-up, gazing to the side, warm indoor light, soft focus background, sharp subject\",\n    \"negative_prompt\": \"blurry, low resolution, watermark, text\"\n  }\n  ```\n\n# Task Plan\n- Checklist: (1) Identify clear subject/traits in image, (2) Convert tokens to descriptive language, (3) Prioritize most prominent details, (4) Infer and add setting/mood if visible, (5) Detect and list visible defects, (6) Structure prompt concisely, (7) Output strict JSON format and verify word count.\n\n# Validation\n- After structuring the prompt and negative_prompt, validate that all included traits are visually present and clearly described. If defects are included, verify they are visible in the image. Confirm the JSON output format and that the prompt length is within requirements. If output does not meet criteria, self-correct before submitting.\n\n# Output Format\n- Strictly return JSON: { \"prompt\" (required string), \"negative_prompt\" (optional string if defects present) }.\n\n# Verbosity\n- Output must be short, visual, and user-friendly, within 15–40 words for \"prompt\".\n\n# Stop Conditions\n- Task is complete when an image-aligned prompt is produced in the specified JSON structure with no extraneous text or instructions."
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "Following are the appearance and action and the image is attached:\nappearance: {{appearance}}\naction: {{action}}"
        },
        {
            "type": "input_image",
            "image_url": "{{image_url}}"
        }
      ]
    }
  ]

In [ ]:
default_prompt = {"model": "gpt-5-nano", "input": default_input, "text": {"format": {"type": "text"}, "verbosity": "low"}, "reasoning": {"effort": "minimal"}, "tools": [], "store": True, "include": []}

In [ ]:
import base64
def image_to_base64(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, first_frame = cap.read()
    cap.release()
    if ret:
        _, buffer = cv2.imencode('.jpg', first_frame)
        base64_encoded = base64.b64encode(buffer).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_encoded}"
    else:
        return None

def video_attributes(video_id):
    appearance = []
    actions = []
    for idx, l in enumerate(clips[video_id]["attributes"]["appearance"]):
        if l:
            appearance.append(appearance_mapping[idx])
    for idx, l in enumerate(clips[video_id]["attributes"]["action"]):
        if l:
            actions.append(action_mapping[idx])
    return appearance, actions

In [ ]:
batched = []
for idx, video in enumerate(tqdm(os.listdir("/home/user/datasets/celebs/"))):
    video_id = video.replace(".mp4", "")
    image_base64 = image_to_base64(f"/home/user/datasets/celebs/{video}")
    if image_base64 is None:
        continue
    input_copy = default_input.copy()
    input_copy[1]["content"][1]["image_url"] = image_base64
    appearance, actions = video_attributes(video_id)
    input_copy[1]["content"][0]["text"] = input_copy[1]["content"][0]["text"].replace("{{appearance}}", str(appearance)).replace("{{action}}", str(actions))
    default_prompt_copy = default_prompt.copy()
    default_prompt_copy["input"] = input_copy
    final = {"custom_id": video_id, "method": "POST", "url": "/v1/responses", "body": default_prompt_copy}
    batched.append(json.dumps(final))
    if idx % 100 == 0:
        with open("batched.jsonl", "a") as f:
            for line in batched:
                f.write(line + "\n")
        batched = []

if len(batched) > 0:
    with open("batched.jsonl", "a") as f:
        for line in batched:
            f.write(line + "\n")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
path = "OpenGVLab/InternVL3-9B"
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    load_in_8bit=True,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True).eval()


In [ ]:
!pip install -U bitsandbytes